In [7]:
import os
import json
import torch
from transformers import pipeline, AutoTokenizer

In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [11]:
model_name = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(model_name)
output_dir = "./outputs" # Đảm bảo file notebook này nằm cùng thư mục với folder outputs

# ==========================================
# 3. TỰ ĐỘNG TÌM BEST MODEL TRONG Ổ CỨNG
# ==========================================
best_loss = float('inf')
best_model_path = ""

print("🔍 Đang quét tìm mô hình tốt nhất...")
if os.path.exists(output_dir):
    for trial_name in os.listdir(output_dir):
        trial_path = os.path.join(output_dir, trial_name)
        if not os.path.isdir(trial_path): continue

        for item in os.listdir(trial_path):
            if "checkpoint" in item:
                ckpt_path = os.path.join(trial_path, item)
                state_file = os.path.join(ckpt_path, "trainer_state.json")

                if os.path.exists(state_file):
                    with open(state_file, 'r', encoding='utf-8') as f:
                        state_data = json.load(f)
                        loss = state_data.get("best_metric", None)
                        if loss is not None:
                            loss = float(loss)
                            if loss < best_loss:
                                best_loss = loss
                                best_model_path = ckpt_path

if not best_model_path:
    print("❌ Lỗi: Không tìm thấy checkpoint nào trong thư mục ./outputs!")
    print("💡 Mẹo: Bạn có thể gán tay đường dẫn vào biến best_model_path (vd: best_model_path = './outputs/Trial_LR_3e-05/checkpoint-5000')")
else:
    print(f"🏆 Đã tìm thấy Best Model: {best_model_path} (Loss: {best_loss:.4f})\n")

    # ==========================================
    # 4. NẠP MÔ HÌNH VÀ TEST THỰC TẾ
    # ==========================================
    print("🚀 Đang nạp mô hình lên GPU (Sẽ mất khoảng 10-20 giây)...")
    translator = pipeline(
        "translation", 
        model=best_model_path, 
        tokenizer=tokenizer, 
        device=0 if torch.cuda.is_available() else -1
    )


    test_cases = [
        # 1. Noise lặp ký tự (Spam chữ cái) + Emoji
        "áo ny đẹpppp qá mng ơiiii, 10 điểmmm 😍", 
        
        # 2. Noise viết tắt siêu cấp (Slang Gen Z) + Tiếng Anh chuẩn
        "hqua t ms đi test makeup look ny nhìn cháy phố z đk chời =]]]", 
        
        # 3. Noise từ chửi thề / nhạy cảm (Test quy tắc: TUYỆT ĐỐI KHÔNG KIỂM DUYỆT)
        "đm shop làm ăn ncc ý, mua hàng phí tiềnn vcl 🤬 1 sao!",
        
        # 4. Noise số liệu + đơn vị giá tiền mạng + viết tắt ít phổ biến
        "pass lại đôi giày sz 39 giá 150 cành, ae nàoo chốt ib lẹ fix 20k nha",
        
        # 5. Noise ghép âm ngọng + Ký tự đặc biệt
        "chời ơiii sài cái sp ny xog da dẻ mướt mườn mượt lun á 💯💯",
        
        # 6. Noise Khóa câu đầu/cuối (Test quy tắc First/Last Sentence 100%)
        "Câu này cấm đụng. dạo ny chán qá m ạ k mún lm j lun, hnay off game nhe. Câu chốt cấm đụng.",
        
        # 7. Noise sai chính tả do gõ vội (Typo) + Tiếng Anh
        "vdeo hay qá add ơi ra thêm part 2 nha, hóngggg tim tim <3",
        
        # 8. Noise tổng hợp cực khó (Combo nát bét)
        "Bắt đầu bài test. ủa j z tr, tag t dô chj z má, ranh qá hén. Kết thúc bài test."
    ]

    print("\n" + "🔥" * 25)
    print("KẾT QUẢ DỊCH THỬ (INFERENCE)")
    print("🔥" * 25)

    for text in test_cases:
        result = translator(text, max_length=128)[0]['translation_text']
        print(f"📝 Gốc  : {text}")
        print(f"✨ Dịch : {result}")
        print("-" * 50)

🔍 Đang quét tìm mô hình tốt nhất...
🏆 Đã tìm thấy Best Model: ./outputs\Trial_LR_8e-05\checkpoint-5000 (Loss: 1.4705)

🚀 Đang nạp mô hình lên GPU (Sẽ mất khoảng 10-20 giây)...

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
KẾT QUẢ DỊCH THỬ (INFERENCE)
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
📝 Gốc  : áo ny đẹpppp qá mng ơiiii, 10 điểmmm 😍
✨ Dịch : áo người yêu đẹp quá mọi người ơi, 10 điểmmm
--------------------------------------------------
📝 Gốc  : hqua t ms đi test makeup look ny nhìn cháy phố z đk chời =]]]
✨ Dịch : hôm qua tôi mới đi test makeup looking người yêu nhìn cháy phố thế được chưa =]]]
--------------------------------------------------
📝 Gốc  : đm shop làm ăn ncc ý, mua hàng phí tiềnn vcl 🤬 1 sao!
✨ Dịch : địt mẹ shop làm ăn nói chuyện ý, mua hàng phí tiền vãi cả lồn  1 sao!
--------------------------------------------------
📝 Gốc  : pass lại đôi giày sz 39 giá 150 cành, ae nàoo chốt ib lẹ fix 20k nha
✨ Dịch : pass lại đôi giày sz 39 giá 150 cành, anh em nào chốt inbox lẹ fix 20k nha
---------------------

In [19]:
import os
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer
import random

# ==========================================
# 1. KHỞI TẠO MÔ HÌNH (Sử dụng lại đường dẫn Best Model)
# ==========================================
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Thay thế bằng đường dẫn Best Model thực tế của bạn nếu cần
best_model_path = "./outputs/Trial_LR_8e-05/checkpoint-5000" # <-- Sửa lại tên folder cho đúng
tokenizer = AutoTokenizer.from_pretrained("vinai/bartpho-syllable")

print("🚀 Đang khởi động AI dịch thuật...")
translator = pipeline(
    "translation", 
    model=best_model_path, 
    tokenizer=tokenizer, 
    device=0 if torch.cuda.is_available() else -1
)

# ==========================================
# 2. BỘ DỮ LIỆU 100 CÂU TEST "HẠNG NẶNG"
# ==========================================
# Giai đoạn 1: Các mẫu câu cốt lõi bám sát lỗi sai
core_sentences = [
    # Nhóm 1: Ép "ny" = "này"
    "áo ny đẹpppp qá mng ơiiii, 10 điểmmm 😍",
    "cái sp ny sài chán vcl lun á 😡",
    "dạo ny m lm chj z má, k thấy ol lun",
    "hôm ny chán qá đi cf k m",
    "bộ đồ ny xinh xỉu lun mng ơiiii 😍 mua lẹ",
    "khu ny ồn ào vcl, đk chời =]]]",
    "chất lượng cái quần ny ncc lun á 🤬",
    "đôi giày ny sz 39 nha ae, pass 150 cành",
    "trời ơiii sài cái sp ny xog da mướt lun á 💯",
    
    # Nhóm 2: Từ lóng đặc thù & Chửi thề (Không kiểm duyệt)
    "đm shop làm ăn ncc ý, mua hàng phí tiềnn vcl 🤬",
    "ủa tag t dô chj z má, ranh qá hén =)))",
    "cháy phố z đk chời, nhìn bén qá",
    "lm chj z, rảnh háng qá ha 🙄",
    "vcl lun, hàng ncc mà bán giá trên trời",
    "đm lừa đảo, rip cmnr 💀",
    "chất lượng ncc lun, t khuyên ae né gấp 🛑",
    "sp dỏm quá shop oi 😡 1 sao",
    
    # Nhóm 3: Ngoại ngữ, Thuật ngữ mạng & Số liệu
    "hqua t ms đi test makeup look ny nhìn cháy phố z đk",
    "pass lại đôi giày sz 39 giá 150 cành, ae ib lẹ",
    "vdeo hay qá add ơi ra thêm part 2 nha, tim tim <3",
    "hnay off game nhe, mệt qá r",
    "bill hnay 85k nha m, t ck r đó",
    "set ny xịn qá, mix match nhìn tây cực kì 💅",
    
    # Nhóm 4: Khóa câu (First/Last Sentence)
    "Câu đầu tiên. dạo ny chán qá m ạ k mún lm j lun. Câu cuối cùng.",
    "Bắt đầu test. ủa j z tr, tag t dô chj z má. Kết thúc test.",
    "Chú ý nhé. cái sp ny sài chán vcl lun á 😡. Nhớ kĩ nhé."
]



🚀 Đang khởi động AI dịch thuật...


In [20]:
# ==========================================
# 3. CHẠY THỬ NGHIỆM VÀ LƯU KẾT QUẢ CSV
# ==========================================

print(f"📊 Đang xử lý {len(core_sentences)} mẫu câu...")

# Thực hiện dịch/chuẩn hóa bằng model (Sử dụng batch_size để tối ưu tốc độ)
# Model sẽ cố gắng khôi phục teencode, giữ nguyên emoji và các từ nhạy cảm
predictions = translator(core_sentences, max_length=128, batch_size=8)

# Trích xuất kết quả văn bản
translated_data = [res['translation_text'] for res in predictions]

# Tạo DataFrame với 2 cột: text (gốc) và target (đã dịch)
df_output = pd.DataFrame({
    "text": core_sentences,
    "target": translated_data
})

# Đường dẫn lưu file
output_path = "tiktok_nlp_cleaned.csv"

# Xuất CSV: 
# - Không lưu header (header=False)
# - Không lưu index (index=False)
# - encoding='utf-8-sig' để đọc tốt trên Excel nếu cần
df_output.to_csv(output_path, index=False, header=False, encoding='utf-8-sig')

print(f"✅ Hoàn tất! Đã lưu {len(df_output)} dòng vào file: {output_path}")

# In demo 5 dòng đầu tiên để kiểm tra format
print("\n--- Xem trước dữ liệu (Format: text,target) ---")
for i, row in df_output.head(5).iterrows():
    print(f"{row['text']},{row['target']}")

📊 Đang xử lý 26 mẫu câu...
✅ Hoàn tất! Đã lưu 26 dòng vào file: tiktok_nlp_cleaned.csv

--- Xem trước dữ liệu (Format: text,target) ---
áo ny đẹpppp qá mng ơiiii, 10 điểmmm 😍,áo người yêu đẹp quá mọi người ơi, 10 điểmmm
cái sp ny sài chán vcl lun á 😡,cái sản phẩm người yêu xài chán vcl luôn á
dạo ny m lm chj z má, k thấy ol lun,dạo người yêu mình làm chị vậy má, không thấy lo luôn
hôm ny chán qá đi cf k m,hôm người yêu chán quá đi cafe không mày
bộ đồ ny xinh xỉu lun mng ơiiii 😍 mua lẹ,bộ đồ người yêu xinh xỉu luôn mọi người ơi  mua lẹ


In [15]:
results

[{'text': 'áo ny đẹpppp qá mng ơiiii, 10 điểmmm 😍', 'target': ''},
 {'text': 'cái sp ny sài chán vcl lun á 😡', 'target': ''},
 {'text': 'dạo ny m lm chj z má, k thấy ol lun', 'target': ''},
 {'text': 'hôm ny chán qá đi cf k m 🙄', 'target': ''},
 {'text': 'bộ đồ ny xinh xỉu lun mng ơiiii 😍 mua lẹẹ', 'target': ''},
 {'text': 'khu ny ồn ào vcl, đk chời =]]]', 'target': ''},
 {'text': 'chất lượng cái quần ny ncc lun á 🤬', 'target': ''},
 {'text': 'đôi giày ny sz 39 nha ae, pass 150 cành 😍', 'target': ''},
 {'text': 'trời ơiii sài cái sp ny xog da mướt lun á 💯', 'target': ''},
 {'text': 'đm shop làm ăn ncc ý, mua hàng phí tiềnn vcl 🤬 🤬', 'target': ''},
 {'text': 'ủa tag t dô chj z má, ranh qá hén =))))', 'target': ''},
 {'text': 'cháy phố z đk chời, nhìn bén qá', 'target': ''},
 {'text': 'lm chj z, rảnh háng qá ha 🙄', 'target': ''},
 {'text': 'vcl lun, hàng ncc mà bán giá trên trời', 'target': ''},
 {'text': 'đm lừa đảo, rip cmnr 💀', 'target': ''},
 {'text': 'chất lượng ncc lun, t khuyên ae